In [1]:
import pandas as pd
import requests
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error

In [2]:
def download_dataset(url, file):
    if not os.path.exists(file):
        parent_dir = os.path.dirname(file)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)
        response = requests.get(url)
        response.raise_for_status()
        with open(file, 'wb') as f:
            f.write(response.content)
    return file

In [3]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df['duration'] = pd.to_datetime(df.tpep_dropoff_datetime) - pd.to_datetime(df.tpep_pickup_datetime)
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    Categorical = ['PULocationID', 'DOLocationID']
    df[Categorical] = df[Categorical].astype(str)
    return df

In [4]:
df_train = read_dataframe(download_dataset('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet', 'yellow_tripdata_2023-01.parquet'))
df_val = read_dataframe(download_dataset('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet', 'yellow_tripdata_2023-02.parquet'))

In [5]:
categorical = ['PULocationID', 'DOLocationID']
numeric = ['trip_distance']
target = ['duration']
train_dict = df_train[categorical + numeric].to_dict(orient = 'records')
val_dict = df_val[categorical + numeric].to_dict(orient = 'records')
vec = DictVectorizer()
x_train = vec.fit_transform(train_dict)
y_train = df_train[target].values
x_val = vec.transform(val_dict)
y_val = df_val[target].values

In [7]:
lr = LinearRegression().fit(x_train, y_train)
y_pred = lr.predict(x_val)
root_mean_squared_error(y_val, y_pred)

7.820279985323501